In [1]:
from openai import OpenAI
import json, time
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
client = OpenAI()

def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


In [ ]:

PRODUCT_CONFIGS = {

    "saas_crm": {
        "product_name": "CRM Platform",
        "categories"  : ["Billing", "Technical", "Feature Request", "Spam", "Other"],
        "urgency"     : ["High", "Medium", "Low"],
        "sla_hours"   : {"High": 2, "Medium": 8, "Low": 24},
        "examples"    : [
            {
                "subject": "SFDC not syncing",
                "body"   : "Salesforce stopped pulling leads.",
                "output" : "Technical"
            },
            {
                "subject": "Charged wrong amount",
                "body"   : "Billed $299 instead of $99.",
                "output" : "Billing"
            }
        ]
    },

    "fintech_payments": {
        "product_name": "Payment Gateway",
        "categories"  : ["Transaction Failed", "Settlement Delay", "KYC Issue", "Fraud Alert", "Other"],
        "urgency"     : ["Critical", "High", "Medium"],
        "sla_hours"   : {"Critical": 1, "High": 4, "Medium": 12},
        "examples"    : [
            {
                "subject": "UPI payment failed",
                "body"   : "UPI transaction deducted but not credited to merchant.",
                "output" : "Transaction Failed"
            },
            {
                "subject": "KYC rejected again",
                "body"   : "Documents rejected for 3rd time. Aadhaar + PAN submitted.",
                "output" : "KYC Issue"
            }
        ]
    },

    "edtech": {
        "product_name": "EdTech Learning Platform",
        "categories"  : ["Course Access", "Payment", "Technical", "Content Query", "Spam", "Other"],
        "urgency"     : ["High", "Medium", "Low"],
        "sla_hours"   : {"High": 4, "Medium": 12, "Low": 48},
        "examples"    : [
            {
                "subject": "Can't access purchased course",
                "body"   : "Paid for the Python course but can't open any videos.",
                "output" : "Course Access"
            },
            {
                "subject": "Certificate not generated",
                "body"   : "Completed the course 3 days ago. Certificate still pending.",
                "output" : "Technical"
            }
        ]
    }
}


In [ ]:
def build_dynamic_prompt(product_key, subject, body):
    """
    Build a product-specific prompt from config.
    Zero hardcoding — everything comes from PRODUCT_CONFIGS.
    """
    config = PRODUCT_CONFIGS[product_key]

    # Build categories list dynamically
    categories_str = "\n".join([f"  - {cat}" for cat in config['categories']])

    # Build examples block dynamically
    examples_str = ""
    for i, ex in enumerate(config['examples'], 1):
        examples_str += f"""
EXAMPLE {i}:
Subject: {ex['subject']}
Body: {ex['body']}
Output: {ex['output']}
"""

    # Build SLA context
    sla_str = ", ".join([
        f"{k}: {v}hr response"
        for k, v in config['sla_hours'].items()
    ])

    return f"""CRITICAL: Return ONLY the category name.

You are an expert support email classifier for {config['product_name']}.

CATEGORIES (pick exactly one):
{categories_str}

URGENCY SLAs: {sla_str}

EXAMPLES FROM THIS PRODUCT:
{examples_str}

NOW CLASSIFY:
Subject: {subject}
Body: {body}

Return ONLY the category name."""

In [25]:
def classify_dynamic(product_key, email):
    prompt = build_dynamic_prompt(
        product_key,
        email['subject'],
        email['body']
    )
    print(prompt)
    return call_llm(prompt)


In [26]:
test_emails = [
    {"product": "saas_crm",        "subject": "Login broken",           "body": "Can't sign in since morning.",              "expected": "Technical"},
    {"product": "fintech_payments", "subject": "Payment not credited",   "body": "Customer paid but order not confirmed.",     "expected": "Transaction Failed"},
    {"product": "edtech",           "subject": "Video not playing",      "body": "Purchased course but videos won't load.",   "expected": "Course Access"},
]

In [27]:
for item in test_emails:
    result = classify_dynamic(item['product'],item)
    print(result)

CRITICAL: Return ONLY the category name.

You are an expert support email classifier for CRM Platform.

CATEGORIES (pick exactly one):
  - Billing
  - Technical
  - Feature Request
  - Spam
  - Other

URGENCY SLAs: High: 2hr response, Medium: 8hr response, Low: 24hr response

EXAMPLES FROM THIS PRODUCT:

EXAMPLE 1:
Subject: SFDC not syncing
Body: Salesforce stopped pulling leads.
Output: Technical

EXAMPLE 2:
Subject: Charged wrong amount
Body: Billed $299 instead of $99.
Output: Billing


NOW CLASSIFY:
Subject: Login broken
Body: Can't sign in since morning.

Return ONLY the category name.
Technical
CRITICAL: Return ONLY the category name.

You are an expert support email classifier for Payment Gateway.

CATEGORIES (pick exactly one):
  - Transaction Failed
  - Settlement Delay
  - KYC Issue
  - Fraud Alert
  - Other

URGENCY SLAs: Critical: 1hr response, High: 4hr response, Medium: 12hr response

EXAMPLES FROM THIS PRODUCT:

EXAMPLE 1:
Subject: UPI payment failed
Body: UPI transactio

In [30]:
## COT vs without COT
email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}

subject = email['subject']
body= email['body']

PROMPT_CURRENT = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""


result = call_llm(PROMPT_CURRENT)

In [31]:
result

'Technical | 9'

In [ ]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other


Subject : {subject}
Body    : {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""


In [ ]:
result = call_llm(PROMPT_COT_V2)
print(result) 

THINKING:
1. Surface complaint: Can't login.
2. Root cause: The issue is likely related to account activation or a problem with the login credentials after the recent payment for the annual plan.
3. Team that OWNS this: Billing team.
4. Business impact: The entire team is unable to access the service, which could hinder their ability to prepare for an important board demo in 3 hours, potentially affecting business operations and client relationships.

CATEGORY: Billing


In [ ]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

See the below examples
email : "I made the payment but did not received the payment confirmation"
category : dummy_category

email : "I made the payment but did not received the payment confirmation"
category : dummy_category

email : "I made the payment but did not received the payment confirmation"
category : dummy_category


Subject : {subject}
Body    : {body}

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""


result = call_llm(PROMPT_COT_V2)


In [47]:
warmup_prompt = """You are a cricket support analyst. 
    
Before I give you a real email to classify, I want you to demonstrate 
your reasoning ability. Generate 3 different example emails a fintech 
support team might receive, and show step-by-step how you would 
classify each one. 

Cover different categories: payment failures, KYC issues, and fraud alerts.

Format each as:
EXAMPLE [N]:
Email: [email text you invent]
Reasoning: [step-by-step thinking]
Classification: [category + urgency]"""


self_examples = call_llm(warmup_prompt)

In [48]:
self_examples

'EXAMPLE 1:\nEmail: "Subject: Payment Failure - Urgent!  \nDear Support Team,  \nI attempted to make a payment of $150 for my recent purchase, but I received an error message stating that the transaction could not be processed. This has caused a delay in my order, and I need this resolved as soon as possible. Please let me know what steps I need to take to complete this payment.  \nThank you,  \nJohn Doe"\n\nReasoning: \n1. The email mentions a specific issue with a payment transaction, indicating a problem with processing a payment.\n2. The urgency is highlighted by the use of the word "Urgent!" in the subject line and the request for a quick resolution.\n3. The email does not mention any personal identification or verification issues, nor does it raise concerns about fraud.\n\nClassification: Payment Failure + Urgent\n\n---\n\nEXAMPLE 2:\nEmail: "Subject: KYC Verification Needed  \nHello,  \nI recently opened an account with your platform, but I received a notification that my KYC (K

In [ ]:
email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}

subject = email['subject']
body= email['body']

In [44]:
classify_prompt = f"""You are a fintech support analyst.

You just demonstrated your reasoning ability with these examples:
{self_examples}

Now apply the SAME reasoning pattern to this real email:

Subject: {subject}
Body: {body}

Follow the same format: Reasoning → Classification JSON."""

In [45]:
classify_prompt

'You are a fintech support analyst.\n\nYou just demonstrated your reasoning ability with these examples:\nEXAMPLE 1:\nEmail: "Hello, I attempted to make a payment of $150 for my subscription, but I received an error message stating that the transaction failed. Can you please help me resolve this issue? Thank you!"\nReasoning: \n1. The email mentions a specific issue related to a payment attempt.\n2. The user describes an error message received during the transaction process.\n3. The request for assistance indicates that the user is seeking a resolution to a problem that is preventing them from completing a payment.\n4. The urgency is moderate, as the user is likely looking to resolve the issue quickly to access their subscription.\nClassification: Payment Failure + Moderate Urgency\n\n---\n\nEXAMPLE 2:\nEmail: "Dear Support Team, I recently submitted my KYC documents, but I received a notification that my account is still not verified. Can you please check the status of my application?

In [46]:
self_examples

'EXAMPLE 1:\nEmail: "Hello, I attempted to make a payment of $150 for my subscription, but I received an error message stating that the transaction failed. Can you please help me resolve this issue? Thank you!"\nReasoning: \n1. The email mentions a specific issue related to a payment attempt.\n2. The user describes an error message received during the transaction process.\n3. The request for assistance indicates that the user is seeking a resolution to a problem that is preventing them from completing a payment.\n4. The urgency is moderate, as the user is likely looking to resolve the issue quickly to access their subscription.\nClassification: Payment Failure + Moderate Urgency\n\n---\n\nEXAMPLE 2:\nEmail: "Dear Support Team, I recently submitted my KYC documents, but I received a notification that my account is still not verified. Can you please check the status of my application? I need to access my funds urgently."\nReasoning:\n1. The email discusses KYC (Know Your Customer) docume

In [50]:
email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}

In [ ]:

PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Important:
- Root cause decides category, not surface words
- Pricing complaint → always Billing
- Feature requests alongside pricing → Billing wins
- 403 errors after payment failure → Billing not Technical

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""


results = []

for i in range(10):
    result = call_llm(PROMPT_COT_V2)
    for line in result.split("\n"):
        if "CATEGORY" in line:
            category = line.replace("CATEGORY:", "").strip()
            results.append(category)
            print(f"Run {i+1} : {category}")

from collections import Counter
votes  = Counter(results)
winner = votes.most_common(1)[0]

for category, count in votes.items():
    bar = "##" * count
    print(f"  {category:20} : {bar} {count}/5")


print(f"Winner     : {winner[0]}")
print(f"Confidence : {winner[1]/5*100:.0f}%")

Run 1 : Technical
Run 2 : Technical
Run 3 : Billing
Run 4 : Technical
Run 5 : Technical
  Technical            : ######## 4/5
  Billing              : ## 1/5
Winner     : Technical
Confidence : 80%


In [52]:
email = {
    "subject": "Evaluating if we stay on your platform",
    "body"   : "We've been using your product for 2 years. "
               "Lately evaluating alternatives. "
               "Main blockers: No Zapier, no bulk export, "
               "pricing jumped 40% at last renewal. "
               "Can we discuss?",
}

subject = email['subject']
body    = email['body']




In [53]:
PROMPT_TOT = f"""You are a Senior Customer Success Manager.

A customer sent this email:
Subject : {subject}
Body    : {body}

Explore 3 different response strategies.
For each — write the response AND score it.

STRATEGY 1: Immediate Discount Offer
Response: [write it]
Score: X/10
Why: [one line]

STRATEGY 2: Empathetic + Schedule a Call
Response: [write it]
Score: X/10
Why: [one line]

STRATEGY 3: Escalate to Account Manager
Response: [write it]
Score: X/10
Why: [one line]

BEST STRATEGY: [which one and why]

FINAL RESPONSE:
[write the winning response]"""


In [54]:
result = call_llm(PROMPT_TOT)
print(result)

**STRATEGY 1: Immediate Discount Offer**  
Response:  
Subject: Re: Evaluating if we stay on your platform  

Hi [Customer's Name],  

Thank you for reaching out and sharing your concerns. I understand that the recent price increase and the lack of features like Zapier and bulk export are significant blockers for you. To show our appreciation for your loyalty over the past two years, I’d like to offer you a 20% discount on your next renewal.  

Please let me know if this helps, and I’d be happy to discuss further.  

Best,  
[Your Name]  
Senior Customer Success Manager  

Score: 6/10  
Why: While the discount may provide immediate relief, it does not address the underlying issues or foster a deeper conversation about the customer's needs.

---

**STRATEGY 2: Empathetic + Schedule a Call**  
Response:  
Subject: Re: Evaluating if we stay on your platform  

Hi [Customer's Name],  

Thank you for your email. I truly appreciate your honesty and understand how frustrating it can be to fac

In [ ]:
config = PRODUCT_CONFIGS['fintech_payments']

categories_str = ".\n".join([cat for cat in config['categories']])
categories_str



# categories = ['Billing', 'Technical', 'Feature Request', 'Spam', 'Other']

'Transaction Failed.\nSettlement Delay.\nKYC Issue.\nFraud Alert.\nOther'

In [14]:
email = 'I paid my emi 2 times'

In [18]:
prompt_saas = f"""

You are email classfier. classify the below email : {email}
and category should be 
- SPAM
- Billing issue

"""

prompt_fin = f"""

You are email classfier. classify the below email : {email}
and category should be 
- Tx issue
- EMI issue

"""

In [19]:
call_llm(prompt_fin)

'The email can be classified under the category: **EMI issue**.'

In [ ]:
# lost in middle
email = {
    "subject": "Re: Re: Fwd: Account Issue - URGENT",
    "body"   : "Hi, following up on my earlier emails. "
               "To summarize the situation: "
               "1) Last month our payment failed but we were still charged. "
               "2) Since then our API access has been intermittent. "
               "3) Our dev team says the SDK throws 403 errors randomly. "
               "4) We tried resetting API keys but same issue. "
               "5) Meanwhile we requested a Zapier integration 2 weeks ago. "
               "6) Our contract renews next month and we are evaluating "
               "whether to continue. "
               "Please escalate this to the right team immediately.",
    "correct": "Billing"
}
